# Segmentação de Imagens: K-Means vs SLIC
### Disciplina de Visão Computacional
Comparação entre o K-Means clássico e o SLIC (Simple Linear Iterative Clustering) para segmentação de imagens, com métricas quantitativas IoU e Dice contra máscara de referência Otsu.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters
from skimage.segmentation import slic, mark_boundaries
from sklearn.cluster import KMeans
import time
import os
import pandas as pd
from IPython.display import display

In [ ]:
def carregar_imagens():
    """
    Carrega imagens de teste a partir de arquivos locais (coins.png e building.jpg).
    Caso os arquivos não existam, usa skimage.data como fallback automático.
    Retorna lista de tuplas (nome_arquivo, imagem_rgb).
    """
    fontes = [
        ('coins.png',    'coins'),
        ('building.jpg', 'brick'),
    ]
    pares = []
    for nome_arquivo, attr_data in fontes:
        if os.path.exists(nome_arquivo):
            img = cv2.imread(nome_arquivo)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            cinza = getattr(data, attr_data, data.camera)()
            img = np.stack([cinza] * 3, axis=-1)
        pares.append((nome_arquivo, img))
    return pares


imagens = carregar_imagens()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nome, img) in zip(axes, imagens):
    ax.imshow(img)
    ax.set_title(nome, fontsize=12)
    ax.axis('off')
plt.suptitle('Imagens de Entrada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

for nome, img in imagens:
    print(f'{nome}: shape={img.shape}, dtype={img.dtype}')

In [ ]:
def aplicar_kmeans(img, k=2):
    """
    Aplica K-Means à imagem em escala de cinza.
    Parâmetros: img (RGB ndarray), k (número de clusters).
    Retorna máscara binária uint8 com o cluster mais brilhante como foreground.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img.copy()
    pixels = gray.reshape(-1, 1).astype(np.float32)
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = km.fit_predict(pixels).reshape(gray.shape)
    idx_fg = int(np.argmax(km.cluster_centers_.flatten()))
    return (rotulos == idx_fg).astype(np.uint8)


def aplicar_slic(img, n_segments=200, compactness=10):
    """
    Aplica SLIC para geração de superpixels e binariza usando limiar de Otsu.
    Cada superpixel cuja intensidade média supera o limiar de Otsu é marcado como foreground.
    Parâmetros: img (RGB ndarray), n_segments (superpixels desejados), compactness (balanço cor/espaço).
    Retorna (mascara_binaria uint8, rotulos_superpixels ndarray).
    """
    img_float = img.astype(np.float64) / 255.0
    rotulos = slic(img_float, n_segments=n_segments, compactness=compactness,
                   start_label=0, channel_axis=-1)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    limiar = filters.threshold_otsu(gray)
    mascara = np.zeros(gray.shape, dtype=np.uint8)
    for lbl in np.unique(rotulos):
        if gray[rotulos == lbl].mean() > limiar:
            mascara[rotulos == lbl] = 1
    return mascara, rotulos


def iou(A, B):
    """
    Calcula Intersection over Union entre duas máscaras binárias.
    Retorna float em [0, 1].
    """
    A, B = A.astype(bool), B.astype(bool)
    inter = np.logical_and(A, B).sum()
    uniao = np.logical_or(A, B).sum()
    return float(inter / uniao) if uniao > 0 else 0.0


def dice(A, B):
    """
    Calcula o Coeficiente Dice entre duas máscaras binárias.
    Retorna float em [0, 1].
    """
    A, B = A.astype(bool), B.astype(bool)
    inter = np.logical_and(A, B).sum()
    denom = A.sum() + B.sum()
    return float(2 * inter / denom) if denom > 0 else 0.0


print('Funções definidas: aplicar_kmeans, aplicar_slic, iou, dice')

In [ ]:
resultados = []

for nome, img in imagens:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gt = (gray > filters.threshold_otsu(gray)).astype(np.uint8)

    t0 = time.perf_counter()
    mask_km_raw = aplicar_kmeans(img, k=2)
    t_km = (time.perf_counter() - t0) * 1000

    iou_normal = iou(mask_km_raw, gt)
    iou_inv    = iou(1 - mask_km_raw, gt)
    mask_km    = mask_km_raw if iou_normal >= iou_inv else (1 - mask_km_raw)
    iou_km     = max(iou_normal, iou_inv)
    dice_km    = dice(mask_km, gt)

    t0 = time.perf_counter()
    mask_slic, rotulos_slic = aplicar_slic(img, n_segments=200, compactness=10)
    t_slic = (time.perf_counter() - t0) * 1000

    iou_slic  = iou(mask_slic, gt)
    dice_slic = dice(mask_slic, gt)

    resultados.append({'Imagem': nome, 'Método': 'K-Means',
                       'Tempo (ms)': round(t_km, 2),
                       'IoU': round(iou_km, 4),
                       'Dice': round(dice_km, 4)})
    resultados.append({'Imagem': nome, 'Método': 'SLIC',
                       'Tempo (ms)': round(t_slic, 2),
                       'IoU': round(iou_slic, 4),
                       'Dice': round(dice_slic, 4)})

    img_vis = mark_boundaries(img.astype(np.float64) / 255.0,
                              rotulos_slic, color=(1, 0, 0))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Comparação — {nome}', fontsize=14, fontweight='bold')

    axes[0].imshow(img)
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(mask_km, cmap='gray')
    axes[1].set_title(
        f'K-Means  (k=2)\nIoU={iou_km:.3f}  Dice={dice_km:.3f}\n{t_km:.1f} ms',
        fontsize=11)
    axes[1].axis('off')

    axes[2].imshow(img_vis)
    axes[2].set_title(
        f'SLIC  (n_segments=200)\nIoU={iou_slic:.3f}  Dice={dice_slic:.3f}\n{t_slic:.1f} ms',
        fontsize=11)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    print(f"\n{'─'*55}")
    print(f"  Imagem : {nome}")
    print(f"  K-Means → {t_km:7.2f} ms | IoU={iou_km:.4f} | Dice={dice_km:.4f}")
    print(f"  SLIC    → {t_slic:7.2f} ms | IoU={iou_slic:.4f} | Dice={dice_slic:.4f}")

In [ ]:
df = pd.DataFrame(resultados)

print("\n" + "="*60)
print("         TABELA COMPARATIVA FINAL: K-Means vs SLIC")
print("="*60)
print(df.to_string(index=False))
print("="*60)

display(df.style
        .set_caption("Comparação Quantitativa: K-Means vs SLIC")
        .format({'Tempo (ms)': '{:.2f}', 'IoU': '{:.4f}', 'Dice': '{:.4f}'})
        .set_properties(**{'text-align': 'center'})
        .highlight_max(subset=['IoU', 'Dice'], color='lightgreen')
        .highlight_min(subset=['Tempo (ms)'], color='lightyellow'))

## Conclusão

### Análise Comparativa: K-Means vs SLIC

O **K-Means** é um algoritmo de agrupamento global baseado em minimização de distâncias no espaço de intensidade. Sua principal vantagem é a simplicidade e velocidade quando o número de clusters é pequeno (k ≤ 5). No entanto, o K-Means não considera a posição espacial dos pixels, o que gera segmentações fragmentadas e sem coerência geográfica — pixels muito distantes podem ser agrupados juntos apenas por terem intensidade similar.

O **SLIC** combina informação de cor/intensidade com coordenadas espaciais no espaço CIELAB, restringindo a busca de cada centroide a uma janela local de tamanho proporcional ao superpixel esperado. O resultado são superpixels compactos que respeitam as bordas naturais dos objetos, propriedade ausente no K-Means global.

---

### Recomendação por Cenário

| Situação | Método Recomendado |
|---|---|
| Preservação de bordas / coerência espacial | **SLIC** |
| Quantização de cor / velocidade máxima | **K-Means** |
| Pré-processamento para deep learning | **SLIC** |
| Prototipagem rápida com k pequeno | **K-Means** |
| Imagens médicas / de satélite | **SLIC** |
| Segmentação por histograma de cor | **K-Means** |

---

> **Conclusão geral:** Para a maioria das aplicações modernas de visão computacional que exigem coerência espacial e respeito às bordas dos objetos, o **SLIC é a escolha superior**. O K-Means permanece relevante para prototipagem rápida, quantização de cor e cenários onde a estrutura espacial da imagem é irrelevante.